# TS2TabPFN: ejecución en Colab

Corre la reproducción de clasificación (TSC) en una sesión con GPU. Es **reanudable**: si la sesión se corta, vuelve a ejecutar todas las celdas y sigue desde donde quedó.

Antes de la primera vez:
1. *Ambiente de execução → Alterar o tipo* → **GPUs: L4**.
2. Acepta los términos de <https://huggingface.co/Prior-Labs/tabpfn_2_5> con tu cuenta de Hugging Face.
3. Crea un token de lectura en Hugging Face y guárdalo en los *Secrets* de Colab (icono de llave) con el nombre **`HF_TOKEN`**, con acceso al notebook activado.

Para usar 2 sesiones en paralelo: abre este notebook dos veces y pon `SHARD = 0` en una y `SHARD = 1` en la otra, con `NUM_SHARDS = 2`.

In [ ]:
# === Parámetros de esta sesión ===
EXTRACTOR = "c22"        # "c22", "tsfresh" o "multirocket"
SHARD = 0                # índice de esta sesión: 0 .. NUM_SHARDS-1
NUM_SHARDS = 1           # cuántas sesiones se reparten el trabajo
DATASETS = []            # vacío = los 134; p. ej. ["ECG200", "ECG5000"] para calibrar
RESAMPLES = "0-29"       # p. ej. "0-2" para calibrar

BRANCH = "phase-1"
REPO_URL = "https://github.com/JEFRY200307/ProyectoDeepSkills.git"
DRIVE_ROOT = "/content/drive/MyDrive/ts2tabpfn"   # datos, features y resultados persistentes
TSC_ZIP_ID = "1p8D-haDDwPKloKMBQCQHgENYHVsjixR4"    # references/TS2TabPFN/README.md

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
for sub in ("data", "features", "results", "logs"):
    os.makedirs(f"{DRIVE_ROOT}/{sub}", exist_ok=True)

In [ ]:
# Código del proyecto (rama BRANCH)
%cd /content
![ -d repo ] || git clone -q -b {BRANCH} {REPO_URL} repo
!git -C repo fetch -q origin {BRANCH} && git -C repo checkout -q {BRANCH} && git -C repo reset -q --hard origin/{BRANCH}
!git -C repo log --oneline -1

In [ ]:
# Entorno: Python 3.10 con las versiones exactas del paper (decisión #9). Tarda unos minutos.
!pip install -q uv==0.9.5
![ -x /content/venv/bin/python ] || uv venv -q --python 3.10 /content/venv
!uv pip install -q --python /content/venv/bin/python -r /content/repo/requirements.txt
!/content/venv/bin/python -c "import tabpfn, aeon, torch; print('tabpfn', tabpfn.__version__, '| aeon', aeon.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')"

In [ ]:
# Token de Hugging Face para los pesos de TabPFN 2.5 (nunca se escribe en el notebook)
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("HF_TOKEN cargado:", bool(os.environ["HF_TOKEN"]))

In [ ]:
# Datos: el zip de los autores se guarda una vez en Drive y se descomprime en el disco local de cada sesión
ZIP = f"{DRIVE_ROOT}/data/tsc_data.zip"
if not os.path.exists(ZIP):
    !/content/venv/bin/python -m gdown {TSC_ZIP_ID} -O {ZIP}
if not os.path.exists("/content/data_tsc"):
    !unzip -q -o {ZIP} -d /content/data_tsc

import glob
hits = glob.glob("/content/data_tsc/**/ECG200/ECG200_TRAIN.ts", recursive=True)
DATA_DIR = os.path.dirname(os.path.dirname(hits[0]))
print("DATA_DIR =", DATA_DIR, "|", len(os.listdir(DATA_DIR)), "carpetas")

In [ ]:
# Ejecución. Cada resample terminado queda en Drive; si se corta, vuelve a correr todo y sigue.
import shlex, subprocess, sys, time

cmd = ["/content/venv/bin/python", "/content/repo/scripts/run_tsc.py",
       "--extractor", EXTRACTOR, "--shard", str(SHARD), "--num-shards", str(NUM_SHARDS),
       "--resamples", RESAMPLES,
       "--data-dir", DATA_DIR,
       "--features-dir", f"{DRIVE_ROOT}/features",
       "--results-dir", f"{DRIVE_ROOT}/results"]
if DATASETS:
    cmd += ["--datasets", *DATASETS]
if EXTRACTOR == "tsfresh":
    cmd += ["--no-extract"]   # las features de tsfresh vienen de AWS

log_path = f"{DRIVE_ROOT}/logs/{EXTRACTOR}_shard{SHARD}_{time.strftime('%Y%m%d_%H%M%S')}.log"
print(" ".join(shlex.quote(c) for c in cmd), "\nlog:", log_path)
with open(log_path, "w") as log:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        sys.stdout.write(line)
        log.write(line)
        log.flush()
print("exit code:", proc.wait())

In [ ]:
# Resumen de lo que hay en Drive hasta ahora
!/content/venv/bin/python /content/repo/scripts/collect_results.py --results-dir {DRIVE_ROOT}/results --out-dir {DRIVE_ROOT}/summary